# Решение тестового задания на позицию "Стажёр Data science"
## Автор: Яковенко Богдан Елисеевич

## 1. Решить задачу
### Вход: FEN
### Выход: Возможные ходы + вероятности

## 2. Дополнительное задание
### Как будете решать задачу, как сыграет конкретный игрок?




# План ноутбука
### 1. Код инференса для задачи 1.
### 2. Полный пайплайн решения задачи 1.  
### 3. Ответ на вопрос "Как буду решать задачу..."

## 1. Код инференса для задачи 1

## Инструкция:
### Необходимо скачать файл model.pt (веса модели) и указать корректный путь до него на вашем компьютере.

Ссылка на скачивание: https://drive.google.com/file/d/1xVl1hbQxKoQgzT6YN6sCpyvihS2LQFib/view?usp=sharing
### Запустить код (достаточно CPU)


In [ ]:
path_to_model = 'Hybrid.pt'
# возможно потребует вашей корректировки

In [ ]:
!pip install chess

In [ ]:
import chess
import numpy as np

In [ ]:
import chess
import torch
import pandas as pd
from tqdm import tqdm


# === 1. Кодирование позиции (AlphaZero style) ===
def fen_to_tensor(fen: str):
    board = chess.Board(fen)
    tensor = torch.zeros((12, 8, 8), dtype=torch.float32)

    piece_map = {
        chess.PAWN: 0,
        chess.KNIGHT: 1,
        chess.BISHOP: 2,
        chess.ROOK: 3,
        chess.QUEEN: 4,
        chess.KING: 5
    }

    for square, piece in board.piece_map().items():
        rank = 7 - (square // 8)
        file = square % 8
        base = piece_map[piece.piece_type]
        idx = base if piece.color == chess.WHITE else base + 6
        tensor[idx, rank, file] = 1

    return tensor  # [12,8,8]


# === 2. Кодирование хода ===
def move_to_coords(move_str: str):
    xs = ord(move_str[0]) - ord('a')
    ys = int(move_str[1]) - 1
    xe = ord(move_str[2]) - ord('a')
    ye = int(move_str[3]) - 1
    return xs, ys, xe, ye




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# === базовый блок ConvNeXt ===
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path=0.0,
     layer_scale_init_value=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(layer_scale_init_value * torch.ones(dim), requires_grad=True)
        self.drop_path = drop_path

    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)         # [B,H,W,C]
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2)         # back to [B,C,H,W]
        return input + x


# === основной гибрид ===
class ConvNeXtTransformerMoveNet(nn.Module):
    def __init__(self, in_ch=12, embed_dim=256, num_blocks=4,
                 nhead=8, num_layers=2, dim_ff=1024, dropout=0.1):
        super().__init__()

        # --- ConvNeXt encoder ---
        self.stem = nn.Conv2d(in_ch, embed_dim, kernel_size=3, padding=1)
        self.blocks = nn.Sequential(*[ConvNeXtBlock(embed_dim) for _ in range(num_blocks)])
        self.norm = nn.LayerNorm(embed_dim)

        # --- Move embedding ---
        self.move_proj = nn.Linear(4, embed_dim)

        # --- Learnable position embeddings ---
        self.pos_emb = nn.Parameter(torch.zeros(1, 65, embed_dim))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        # --- Transformer head ---
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # --- Classifier ---
        self.fc = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, board, move):
        # ConvNeXt feature extraction
        x = self.stem(board)
        x = self.blocks(x)
        B, C, H, W = x.shape
        x = x.flatten(2).permute(0, 2, 1)      # [B,64,C]
        x = self.norm(x)

        # Move token
        move_token = self.move_proj(move).unsqueeze(1)  # [B,1,C]
        x = torch.cat([x, move_token], dim=1)           # [B,65,C]

        # Add positional embeddings
        x = x + self.pos_emb[:, :x.size(1), :]

        # Transformer reasoning
        x = self.encoder(x)

        # Pool + classifier
        x = x.mean(dim=1)
        out = self.fc(x)
        return out  # логиты


In [ ]:
model = ConvNeXtTransformerMoveNet()       # или твоя архитектура
model.load_state_dict(torch.load(path_to_model, map_location="cpu"))
model.eval()

In [ ]:
import torch
import chess
import chess.engine
import chess.pgn
from tqdm import tqdm




# === инференс ===
def predict_moves(fen: str, model, device='cpu'):
    board_tensor = fen_to_tensor(fen).unsqueeze(0).to(device)  # [1,12,8,8]
    board = chess.Board(fen)

    model.eval()
    preds = []
    with torch.inference_mode():
        for move in board.legal_moves:
            move_vec = torch.tensor(move_to_coords(move.uci()), dtype=torch.float32).unsqueeze(0).to(device)
            logits = model(board_tensor, move_vec)
            prob = torch.sigmoid(logits).item()
            preds.append((move.uci(), prob))

    # сортировка по убыванию вероятности
    preds.sort(key=lambda x: x[1], reverse=True)
    moves = [i[0] for i in preds]
    ps = [i[1] for i in preds]
    ps = [p/sum(ps) for p in ps]
    # вывод
    print(f"\nFEN: {fen}")
    print("Ходы:")
    for m, p in zip(moves, ps):
        print(f"{m:8s}  {p:.4f}")

    return preds


In [ ]:
import pandas as pd
import random
# случайный FEN из колонки 'fen'

fen = 'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1' # Введите свой FEN


print(f"Случайный FEN:\n{fen}")

import time

# замер времени инференса
start = time.perf_counter()
predict_moves(fen, model)
end = time.perf_counter()

print(f"\n⏱ Время инференса: {end - start:.3f} сек")

# 3. Полное решение задачи 1

## Работа с данными

### 1) Добавляем метки 0 или 1. Если fen + move уже были в датасеты - 1, иначе 0.
### 2) Кодируем FEN к виду (12, 8, 8). Почему 12? - 6 типов фигур, 2 цвета. 8 на 8 размер шахматной доски. Такой же формат применяется в AlphaZero (DeepMind).
### 3) Кодируем move как 4 координаты Xs, Ys, Xe, Ye.
### 4) Сохраняем датасет по чанкам (чтобы не грузить VRAM)

In [ ]:
import pandas as pd
df = pd.read_csv("path_to_data")

df = df.drop_duplicates()

In [ ]:
!pip install chess

In [ ]:
import chess
import torch
import pandas as pd
from tqdm import tqdm


# === 1. Кодирование позиции (AlphaZero style) ===
def fen_to_tensor(fen: str):
    board = chess.Board(fen)
    tensor = torch.zeros((12, 8, 8), dtype=torch.float32)

    piece_map = {
        chess.PAWN: 0,
        chess.KNIGHT: 1,
        chess.BISHOP: 2,
        chess.ROOK: 3,
        chess.QUEEN: 4,
        chess.KING: 5
    }

    for square, piece in board.piece_map().items():
        rank = 7 - (square // 8)
        file = square % 8
        base = piece_map[piece.piece_type]
        idx = base if piece.color == chess.WHITE else base + 6
        tensor[idx, rank, file] = 1

    return tensor  # [12,8,8]


# === 2. Кодирование хода ===
def move_to_coords(move_str: str):
    xs = ord(move_str[0]) - ord('a')
    ys = int(move_str[1]) - 1
    xe = ord(move_str[2]) - ord('a')
    ye = int(move_str[3]) - 1
    return xs, ys, xe, ye


# === 3. Генерация всех легальных ходов и лейблов ===
def expand_with_legal_moves(df: pd.DataFrame):
    records = []
    unique_fens = df["fen"].unique().tolist()
    df_grouped = df.groupby("fen")["move"].apply(set).to_dict()

    for fen in tqdm(unique_fens):
        board = chess.Board(fen)
        legal_moves = [m.uci() for m in board.legal_moves]
        played_moves = df_grouped.get(fen, set())

        for move in legal_moves:
            label = 1 if move in played_moves else 0
            records.append((fen, move, label))

    return pd.DataFrame(records, columns=["fen", "move", "label"])

In [ ]:



# === 4. Сборка финальных признаков (flatten board + move) ===
def build_feature_dataset(df: pd.DataFrame, save_path="chess_features_part_K.pt", save_each=500_000):
    features, labels = [], []
    part_num = 1
    n = 0
    for fen, move, label in tqdm(zip(df["fen"], df["move"], df["label"]), total=len(df)):
        try:
            board_tensor = fen_to_tensor(fen)
            move_tensor = torch.tensor(move_to_coords(move), dtype=torch.float32)

            # flatten + concat
            feat = torch.cat([
                board_tensor.flatten(),  # 12*8*8 = 768
                move_tensor              # +4 = 772
            ])

            features.append(feat)
            labels.append(label)

        except Exception:
            continue

        n+=1

        if n%save_each==0:
            features = torch.stack(features)  # [N,772]
            labels = torch.tensor(labels, dtype=torch.float32)

            dataset = {"X": features, "y": labels}
            torch.save(dataset, save_path.replace('K', str(part_num)))
            print(f"Saved {len(features)} samples → {save_path.replace('K', str(part_num))}")

            part_num+=1
            features, labels = [], []


In [ ]:
chunk_size = 250000

In [ ]:
expanded = expand_with_legal_moves(df)

In [ ]:
expanded = expanded.iloc[:chunk_size*6]

In [ ]:
expanded['label'].value_counts()

In [ ]:
build_feature_dataset(expanded, save_each=chunk_size)

In [ ]:
vc=expanded['label'].value_counts().reset_index()

In [ ]:
vc

In [ ]:
scale_pos_weight = vc['count'][0]/vc['count'][1]
# позже будет использоваться для нейтрализации дисбаланса в данных

In [ ]:
base_path = 'chess_features_part_K.pt'
N_PARTS = len(expanded)//chunk_size+(1-len(expanded)%chunk_size==0)

In [ ]:
# expanded.to_parquet("expanded_chess.parquet")

In [ ]:
!pip install optuna

In [ ]:
import torch
import xgboost as xgb
import optuna
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

## XGBoost + GPU
## Otpuna для подбора параметров

In [ ]:
# === 1. Настройки ===

N_TRIALS = 20                 # Optuna n_trials
NUM_BOOST_ROUND_PER_PART = 100
EARLY_STOPPING = 25

In [ ]:

ml_N_PART = 2 # берём поменьше, т.к. классические методы не требуют такой объём данных

# === 2. Objective для Optuna ===
def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "gpu_id": 0,
        "verbosity": 0,

        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "scale_pos_weight": scale_pos_weight
    }

    booster = None
    all_val_preds, all_val_labels = [], []

    # === 3. Итеративно обучаем на кусках ===
    for i in tqdm(range(ml_N_PART), desc=f"Trial {trial.number}"):
        ds = torch.load(base_path.replace('K', str(i+1)))
        X = ds["X"].numpy().astype("float32")
        y = ds["y"].numpy().astype("float32")

        # Разделим кусок на train/val внутри (например 90/10)
        split = int(len(X) * 0.9)
        X_train, X_val = X[:split], X[split:]
        y_train, y_val = y[:split], y[split:]

        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval   = xgb.DMatrix(X_val, label=y_val)

        evals = [(dtrain, "train"), (dval, "val")]

        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=NUM_BOOST_ROUND_PER_PART,
            evals=evals,
            early_stopping_rounds=EARLY_STOPPING,
            verbose_eval=False,
            xgb_model=booster  # <== продолжаем обучение
        )

        # накопим предсказания на валидации
        preds = booster.predict(dval)
        all_val_preds.extend(preds)
        all_val_labels.extend(y_val)

        # чистим память
        del ds, X, y, X_train, X_val, y_train, y_val, dtrain, dval

    # === 4. Считаем общую метрику ===
    auc = roc_auc_score(all_val_labels, all_val_preds)
    return auc





In [ ]:
    print("Starting Optuna tuning with incremental GPU training...")
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
    study.optimize(objective, n_trials=N_TRIALS)

    print("\nBest trial params:")
    print(study.best_trial.params)
    print(f"Best AUC = {study.best_value:.5f}")

    # сохраняем для повторного обучения на полном наборе
    study.trials_dataframe().to_csv("optuna_results.csv", index=False)


## LogReg + GPU
## Otpuna для подбора параметров

In [ ]:
import torch
import optuna
import numpy as np
from tqdm import tqdm
from cuml.linear_model import LogisticRegression
from cuml.metrics import roc_auc_score as gpu_roc_auc_score
import cupy as cp





def objective(trial):
    C = trial.suggest_float("C", 1e-3, 100.0, log=True)
    penalty = trial.suggest_categorical("penalty", ["l2", None])
    max_iter = 1000
    tol = 1e-4

    all_preds, all_labels = [], []

    for i in tqdm(range(ml_N_PART), desc=f"Trial {trial.number}"):
        ds = torch.load(base_path.replace("K", str(i + 1)))
        X = ds["X"].numpy().astype("float32")
        y = ds["y"].numpy().astype("float32")

        split = int(len(X) * 0.9)
        X_train, X_val = X[:split], X[split:]
        y_train, y_val = y[:split], y[split:]

        # CPU → GPU (cupy)
        X_train_gpu = cp.asarray(X_train)
        X_val_gpu = cp.asarray(X_val)
        y_train_gpu = cp.asarray(y_train)
        y_val_gpu = cp.asarray(y_val)

        # === GPU Logistic Regression ===
        model = LogisticRegression(
            C=C,
            penalty=penalty,
            max_iter=max_iter,
            tol=tol,
            fit_intercept=True,
            class_weight={0: 1.0, 1: scale_pos_weight},
            verbose=0
        )

        model.fit(X_train_gpu, y_train_gpu)
        preds = model.predict_proba(X_val_gpu)[:, 1]

        all_preds.append(preds)
        all_labels.append(y_val_gpu)

        del ds, X, y, X_train, X_val, y_train, y_val, X_train_gpu, X_val_gpu, y_train_gpu, y_val_gpu, model

    # объединяем батчи на GPU
    all_preds = cp.concatenate(all_preds)
    all_labels = cp.concatenate(all_labels)

    auc = float(gpu_roc_auc_score(all_labels, all_preds))
    return auc


print("Optuna tuning Logistic Regression (cuML GPU)...")
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=N_TRIALS)

print("\nBest trial params:")
print(study.best_trial.params)
print(f"Best AUC = {study.best_value:.5f}")

study.trials_dataframe().to_csv("optuna_logreg_gpu_results.csv", index=False)


## Train pipeline (networks)

In [ ]:
def train_pipeline(model, save_path):
    EPOCHS = 50
    BATCH_SIZE = 1024
    LR = 3e-4
    VAL_RATIO = 0.1
    patience = 2
    best_metric = 0

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(scale_pos_weight, device=device))

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)


    # === цикл ===
    for epoch in range(EPOCHS):
        print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")
        model.train()
        running_loss = 0.0

        all_val_preds, all_val_labels = [], []

        for part in range(1, N_PARTS + 1):
            ds = torch.load(base_path.replace("K", str(part)))
            X = ds["X"]
            y = ds["y"].float().unsqueeze(1)

            # разбор признаков
            board_flat = X[:, :768].view(-1, 12, 8, 8)
            move_vec = X[:, 768:]

            full_dataset = TensorDataset(board_flat, move_vec, y)
            val_size = int(len(full_dataset) * VAL_RATIO)
            train_size = len(full_dataset) - val_size

            train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

            # === train ===
            model.train()
            for board, move, labels in tqdm(train_loader, desc=f"Train part {part}", leave=False):
                board, move, labels = board.to(device), move.to(device), labels.to(device)
                preds = model(board, move)
                loss = criterion(preds, labels)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * board.size(0)

            # === validation ===
            model.eval()
            with torch.no_grad():
                preds_all, labels_all = [], []
                for board, move, labels in val_loader:
                    board, move = board.to(device), move.to(device)
                    preds = model(board, move).detach().cpu()
                    preds_all.append(preds)
                    labels_all.append(labels)

                preds_all = torch.cat(preds_all).numpy()
                labels_all = torch.cat(labels_all).numpy()
                val_auc = roc_auc_score(labels_all, preds_all)

                all_val_preds.extend(preds_all)
                all_val_labels.extend(labels_all)

            print(f"Part {part}: val_auc={val_auc:.4f}")

            del ds, X, y, board_flat, move_vec, full_dataset, train_ds, val_ds, train_loader, val_loader

        # итог по эпохе
        avg_loss = running_loss / (N_PARTS * (1 - VAL_RATIO) * 500_000)
        epoch_auc = roc_auc_score(all_val_labels, all_val_preds)
        if best_metric>epoch_auc:
          patience-=1
          if patience==0:
            print('Early stopping!')
            break
        else:
          best_metric=epoch_auc
          torch.save(model.state_dict(), save_path)
          print(f"✅ Saved new best model")
        print(f"Epoch {epoch+1}: loss={avg_loss:.6f}, val_auc={epoch_auc:.4f}")


    print(f'Итог. Лучший val_loss {best_metric}')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

## Custom CNN
## Архитекутра на основе ResidulBlock (ResNet)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += identity
        return F.relu(out)


class ResCNNMoveNet(nn.Module):
    def __init__(self, in_channels=12, move_dim=4, base_channels=64, fc_dim=256):
        super().__init__()

        # === move embedding ===
        self.move_embed = nn.Sequential(
            nn.Linear(move_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU()
        )

        # === CNN trunk ===
        self.conv1 = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(base_channels)

        self.layer1 = self._make_layer(base_channels, num_blocks=2)
        self.layer2 = self._make_layer(base_channels * 2, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(base_channels * 4, num_blocks=2, stride=2)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # === классификатор ===
        self.fc = nn.Sequential(
            nn.Linear(base_channels * 4 + 256, fc_dim),
            nn.BatchNorm1d(fc_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(fc_dim, 1)
        )

    def _make_layer(self, out_channels, num_blocks, stride=1):
        layers = []
        layers.append(nn.Conv2d(out_channels // 2 if stride != 1 else out_channels,
                                out_channels, kernel_size=3, stride=stride,
                                padding=1, bias=False))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        for _ in range(num_blocks):
            layers.append(ResidualBlock(out_channels))
        return nn.Sequential(*layers)

    def forward(self, board, move):
        # move: [B, 4]
        move_feat = self.move_embed(move)

        # board: [B, 12, 8, 8]
        x = F.relu(self.bn1(self.conv1(board)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.global_pool(x).flatten(1)  # [B, C]

        out = torch.cat([x, move_feat], dim=1)
        out = self.fc(out)
        return torch.sigmoid(out)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResCNNMoveNet().to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
train_pipeline(model, 'CNN.pt')

## Mini-transformer


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class MiniTransformerMoveNet(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=2, dim_feedforward=1024, dropout=0.1):
        super().__init__()

        # === input projections ===
        self.board_proj = nn.Linear(12, d_model)
        self.move_proj = nn.Linear(4, d_model)

        # === learnable positional embeddings ===
        self.pos_emb = nn.Parameter(torch.zeros(1, 65, d_model))  # 64 клетки + 1 move
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        # === transformer encoder ===
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # === нормализация + классификатор ===
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, board, move):
        # board: [B, 12, 8, 8] → [B, 64, 12]
        board_tokens = board.view(board.size(0), 12, 64).permute(0, 2, 1)
        board_emb = self.board_proj(board_tokens)  # [B, 64, d_model]

        # move: [B, 4] → [B, 1, d_model]
        move_emb = self.move_proj(move).unsqueeze(1)

        # объединяем board + move
        x = torch.cat([board_emb, move_emb], dim=1)  # [B, 65, d_model]

        # добавляем обучаемые позиционные вектора
        x = x + self.pos_emb[:, :x.size(1), :]

        # transformer encoder
        x = self.encoder(x)
        x = self.norm(x)

        # mean pooling
        x = x.mean(dim=1)
        out = self.fc(x)
        return out  # логиты, не сигмоида


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MiniTransformerMoveNet().to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
train_pipeline(model, 'Transformer.pt')

## CNN+attention
## Архитектура на основе ConvNext


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# === базовый блок ConvNeXt ===
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, drop_path=0.0,
     layer_scale_init_value=1e-6):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(layer_scale_init_value * torch.ones(dim), requires_grad=True)
        self.drop_path = drop_path

    def forward(self, x):
        input = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)         # [B,H,W,C]
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2)         # back to [B,C,H,W]
        return input + x


# === основной гибрид ===
class ConvNeXtTransformerMoveNet(nn.Module):
    def __init__(self, in_ch=12, embed_dim=256, num_blocks=4,
                 nhead=8, num_layers=2, dim_ff=1024, dropout=0.1):
        super().__init__()

        # --- ConvNeXt encoder ---
        self.stem = nn.Conv2d(in_ch, embed_dim, kernel_size=3, padding=1)
        self.blocks = nn.Sequential(*[ConvNeXtBlock(embed_dim) for _ in range(num_blocks)])
        self.norm = nn.LayerNorm(embed_dim)

        # --- Move embedding ---
        self.move_proj = nn.Linear(4, embed_dim)

        # --- Learnable position embeddings ---
        self.pos_emb = nn.Parameter(torch.zeros(1, 65, embed_dim))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        # --- Transformer head ---
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # --- Classifier ---
        self.fc = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, board, move):
        # ConvNeXt feature extraction
        x = self.stem(board)
        x = self.blocks(x)
        B, C, H, W = x.shape
        x = x.flatten(2).permute(0, 2, 1)      # [B,64,C]
        x = self.norm(x)

        # Move token
        move_token = self.move_proj(move).unsqueeze(1)  # [B,1,C]
        x = torch.cat([x, move_token], dim=1)           # [B,65,C]

        # Add positional embeddings
        x = x + self.pos_emb[:, :x.size(1), :]

        # Transformer reasoning
        x = self.encoder(x)

        # Pool + classifier
        x = x.mean(dim=1)
        out = self.fc(x)
        return out  # логиты


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConvNeXtTransformerMoveNet().to(device)
sum(p.numel() for p in model.parameters())

In [ ]:
train_pipeline(model, 'Hybrid.pt')

## Результаты


| Модель                          | Размер модели | ROC AUC       |
| ------------------------------- | ----------------- | ------------- |
| XGBoost                         | пренебрежимо мал           | 0.72          |
| Логистическая регрессия         | пренебрежимо мал           | 0.61          |
| Custom CNN                      | 3.6 M             | 0.73         |
| Mini-Transformer                | 1.6 M             | 0.746 |
| Hybrid (ConvNeXt + Transformer) | 3.8 M     | 0.95 |


# С отрывом побеждает Hybrid модель. Она же и самая большая модель из представленных. ROC AUC высок, значит предсказания модели будут относительно объективны.

## 2. Дополнительное задание

## Ответ:

### Подход к моделированию шахматного стиля

**1. Этап 1 — Обнаружение индивидуальных стилей**

* Беру партии известных шахматистов и присваиваю каждому **индивидуальный стиль** (“Стиль ФИО”).
* Обучаю **модель 1 — классификатор**, предсказывающий стиль по партии (последовательности состояний досок).
* Если модель достигает **F1 > 0.85**, можно сделать вывод, что связь *«шахматист ↔ стиль»* существует.
* Архитектура: **CNN** или **Attention**.
* Далее беру **скрытое состояние последнего слоя** и усредняю его по партиям игрока → получаю **вектор стиля**.
* Если два вектора похожи — объединяю их в один стиль как среднее между ними.
* Так формируется **набор векторов стилей**.

---

**2. Этап 2 — Идентификация стиля по фрагменту партии**

* Обучаю **модель 2**, которая по **фрагменту** партии (подпоследовательности ходов) предсказывает стиль.
* Размер входа подбирается экспериментально до достижения **стабильной метрики** (ориентир — F1 > 0.85).
* Скрытое состояние последнего слоя трактуется как **вектор стиля игрока**.

---

**3. Этап 3 — Прогноз следующего хода с учётом стиля**

* Обучаю **модель 3**, получающую на вход:

  * текущее **состояние доски**,
  * **вектор стиля игрока**.
* Задача — предсказать **следующий ход**.
* Для обучения используется тот же набор данных, но в формате **отдельных позиций**.
* Вектор игрока — усреднённый по его партиям.

---

**4. Объём данных и устойчивость**

* Для устойчивой оценки стиля требуется **3–4 партии**.
* Для каждой вычисляется вектор, затем они усредняются.
* **Стабильность вектора** зависит от игрока, но диапазон **2–5 партий** реалистичен.

---

**5. Отличие персональной модели от общей**

* После первых **3–4 партий** формируются **базовые векторы стилей**,
  отличные от общей модели, что **повышает точность предсказаний** модели 3.

---
